In [ ]:
# change working directory to the root of the project
%cd ..

## Camera Trajectory Visualization

In [2]:
import plotly.graph_objects as go
import numpy as np

def visualize_cameras_frustum(c2w_dict, points=None, save_path=None):
    """use frustum to visualize camera poses
    """
    fig = go.Figure()
    
    # frustum parameters
    frustum_length = 0.1
    frustum_aspect = 0.75
    frustum_base = 0.08
    
    def create_frustum_vertices(c2w):
        # use opencv camera model
        pos = c2w[:3, 3]
        forward = c2w[:3, 2]  # z-axis is forward
        right = c2w[:3, 0]    # x-axis is right
        up = c2w[:3, 1]       # y-axis is up
        
        # calculate frustum vertices
        apex = pos
        half_width = frustum_base / 2
        half_height = frustum_base * frustum_aspect / 2
        
        # bottom four vertices
        bl = pos + frustum_length * forward - half_width * right - half_height * up
        br = pos + frustum_length * forward + half_width * right - half_height * up
        tl = pos + frustum_length * forward - half_width * right + half_height * up
        tr = pos + frustum_length * forward + half_width * right + half_height * up
        
        return apex, bl, br, tl, tr
    
    def add_camera_frustum(name, c2w, color, group_name):
        """add single camera frustum"""
        apex, bl, br, tl, tr = create_frustum_vertices(c2w)
        
        # draw camera center point
        fig.add_trace(go.Scatter3d(
            x=[apex[0]], y=[apex[1]], z=[apex[2]],
            mode='markers',
            marker=dict(size=1, color=color),
            name=group_name,
            text=name,
            showlegend=False,
        ))
        
        # build frustum vertices sequence
        lines = np.array([
            apex, bl, br, apex, tl, tr, apex,  # from apex to bottom
            bl, tl, tr, br, bl                 # bottom border
        ])
        
        # draw frustum line
        fig.add_trace(go.Scatter3d(
            x=lines[:, 0], y=lines[:, 1], z=lines[:, 2],
            mode='lines',
            line=dict(color=color, width=1),
            name=group_name,
            text=name,
            showlegend=False
        ))

    for c2w in c2w_dict.values():
        add_camera_frustum('cam', c2w, 'blue', 'cam')
    
    if points is not None:
        fig.add_trace(go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode='markers',
            marker=dict(
                size=0.1,
                color='red',
                opacity=0.8,
            ),
            name='Point Cloud'
        ))
    
    # set figure layout
    fig.update_layout(
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data'  # keep real ratio
        ),
        width=1000,
        height=1000,
        title='Camera Poses Visualization'
    )
    if save_path:  # save as html
        fig.write_html(save_path)
        print(f"Visualization saved to {save_path}")
    else:
        fig.show()

In [ ]:
from gs_toolkit.entry import Entry

entry = Entry(scene_path='./data/bicycle', output_path='./output')

In [4]:
from gs_toolkit.utils.render_utils import generate_bounding_trajectory
from gs_toolkit.utils.graphics_utils import getWorld2View2

cameras = generate_bounding_trajectory(entry.cameras, 240)
c2w_dict = {idx:  np.linalg.inv(getWorld2View2(camera.R, camera.T, camera.trans, camera.scale)) for idx, camera in enumerate(cameras)}

point_cloud = entry.gs_model.get_xyz.cpu().numpy()

visualize_cameras_frustum(c2w_dict, point_cloud, save_path='cam_traj.html')